# Logistic Regression and Model Evaluation – Concepts

This notebook summarises the concepts and adds explanations and small examples. All examples use small hand-made numbers – no data set is loaded.

**Contents**
1. Logistic regression: a linear model for classification
2. Fitting by maximum likelihood (log-loss)
3. Confusion matrix and metrics
4. Classification thresholds
5. ROC curve and AUC

## Libraries and settings

In [ ]:
# Libraries
import os
import platform
import warnings
from datetime import datetime
from platform import python_version

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score

# Ignore warnings
warnings.filterwarnings('ignore')

## 1. Logistic regression: a linear model for classification

$$P(y = 1 \mid x) = \sigma(\beta_0 + \beta_1 x) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x)}}$$

In words: a linear **score** $\beta_0 + \beta_1 x$ is squeezed by an S-curve (the *sigmoid* $\sigma$) into a probability
between 0 and 1. With several features the score is $\beta_0 + \beta_1 x_1 + \dots + \beta_p x_p$.

- $\beta_1 > 0$: the probability rises with $x$.
- Example: $\beta_0 = -5$, $\beta_1 = 1$, $x = 6$ → score 1 → $p = 1/(1+e^{-1}) \approx 0.73$ → class 1 (if $\tau = 0.5$).

In [ ]:
# Function: converts a score into a probability between 0 and 1
def sigmoid(s):
    """Sigmoid function: maps a score to a probability."""
    return 1 / (1 + np.exp(-s))


# Example: beta_0 = -5, beta_1 = 1 and x = 6 -> score = 1
print('p =', round(sigmoid(-5 + 1 * 6), 3))

# Figure showing the S-curve for beta_0 = -5 and beta_1 = 1
x = np.linspace(0, 10, 100)
plt.plot(x, sigmoid(-5 + 1 * x))
plt.axhline(0.5, color='orange', linestyle='--', label='threshold 0.5')
plt.xlabel('x')
plt.ylabel('P(y = 1 | x)')
plt.legend()
plt.show()

## 2. Fitting by maximum likelihood (log-loss)

Logistic regression is fitted by **maximum likelihood**: choose $\beta$ so that the observed classes are as plausible as
possible. Maximising the likelihood is the same as minimising the average **log-loss** (cross-entropy):

$$\text{log-loss} = -\frac{1}{n}\sum_{i=1}^n \Big[y_i \log \hat p_i + (1-y_i)\log(1-\hat p_i)\Big]$$

There is no closed-form solution – the coefficients are computed numerically (e.g. by gradient-based methods). A
confident wrong prediction ($\hat p = 0.99$ for a negative case) is penalised heavily.

In [ ]:
# Log-loss of a positive case (y = 1) for different predicted probabilities
for p_hat in [0.9, 0.6, 0.5, 0.1, 0.01]:
    print('y = 1, p_hat =', p_hat, ' loss =', round(-np.log(p_hat), 3))

## 3. Confusion matrix and metrics

| | predicted negative | predicted positive |
|---|---|---|
| **actual negative** | TN | FP |
| **actual positive** | FN | TP |

This is the orientation of scikit-learn: `confusion_matrix(y_true, y_pred)` returns `[[TN, FP], [FN, TP]]` (rows =
actual class, columns = predicted class, ordered 0, 1).

| metric | formula | question |
|---|---|---|
| accuracy | $\frac{TP + TN}{n}$ | share of all predictions that are correct |
| precision | $\frac{TP}{TP + FP}$ | of all predicted positives, how many are really positive? |
| recall (TPR, sensitivity) | $\frac{TP}{TP + FN}$ | of all real positives, how many did we find? |
| FPR | $\frac{FP}{FP + TN}$ | of all real negatives, how many were wrongly flagged? |
| F1 | $2\cdot\frac{\text{precision}\cdot\text{recall}}{\text{precision}+\text{recall}}$ | one number balancing precision and recall |

**Caution:** with imbalanced classes, accuracy alone misleads. Example with $n = 171$ test observations:

In [ ]:
# Example confusion matrix with n = 171 test observations
TP = 50
FP = 22
FN = 24
TN = 75
n = TP + FP + FN + TN

# Metrics
print('accuracy  =', round((TP + TN) / n, 3))
print('precision =', round(TP / (TP + FP), 3))
print('recall    =', round(TP / (TP + FN), 3))
print('FPR       =', round(FP / (FP + TN), 3))

# Imbalanced classes: 1000 cases, only 20 positives, always predict "negative"
print('Always "negative": accuracy =', 980 / 1000, ' recall = 0')

## 4. Classification thresholds

$$\hat y = 1 \quad \text{if} \quad \hat p(x) \ge \tau$$

- $\tau$ ↑: fewer predicted positives – recall and FPR cannot increase; precision often rises, but this is not
  guaranteed.
- $\tau$ ↓: more predicted positives – recall and FPR cannot decrease.
- The default $\tau = 0.5$ is common for probability outputs; score-based classifiers (e.g. SVM) use an analogous
  threshold.

**Rule:** pick $\tau$ from the **costs** of false positives vs. false negatives. If a false negative costs $c$ times as
much as a false positive, the cost-optimal threshold of a well-calibrated model is about $\tau^* = 1/(1+c)$.

In [ ]:
# Ten cases: true class and predicted probability of class 1
y_true = [0, 0, 0, 0, 0, 1, 1, 1, 1, 1]
p_hat = np.array([0.1, 0.2, 0.35, 0.45, 0.6, 0.4, 0.55, 0.7, 0.8, 0.9])

# Confusion matrix [[TN, FP], [FN, TP]] for three thresholds
for threshold in [0.3, 0.5, 0.7]:
    y_pred = np.where(p_hat >= threshold, 1, 0)
    print('Threshold', threshold)
    print(confusion_matrix(y_true, y_pred))

## 5. ROC curve and AUC

**ROC curve:** for every threshold $\tau$ plot (FPR, TPR) and connect the points. Each point is one threshold; the
diagonal corresponds to a random ranking (in expectation).

**AUC** = area under the ROC curve (0.5 = random ranking, 1 = perfect). Interpretation: the **probability that a random
positive case gets a higher score than a random negative case** (ties count ½). Example: AUC = 0.95 → in 95 % of such
pairs the positive case is ranked higher.

In [ ]:
# ROC curve of the same ten cases (one point per threshold) and AUC
fpr, tpr, thresholds = roc_curve(y_true, p_hat)
print('AUC =', roc_auc_score(y_true, p_hat))

# Figure showing the ROC curve
plt.plot(fpr, tpr, marker='o', label='model')
plt.plot([0, 1], [0, 1], 'k--', label='random ranking')
plt.xlabel('false positive rate (FPR)')
plt.ylabel('true positive rate (TPR)')
plt.legend()
plt.show()

**Key messages**
- Logistic regression turns a linear score into a probability; it is fitted by maximum likelihood (log-loss).
- With imbalanced classes look at precision, recall and FPR – not only at the accuracy.
- The threshold is a business decision based on the costs of errors.
- The ROC curve shows all thresholds at once; the AUC measures how well the model ranks positives above negatives.

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')